# DEKM embeddings for the FP-Max study: RCV1, seeds 42-51

Trains DEKM (frozen canonical protocol, `DEKM_CANONICAL_BASELINE.md`) on every row of
RCV1 for seeds 42-51, then exports one **study bundle** per seed: a fixed,
label-free sample of `data.dekm.n_rows` rows with their embedding, tf-idf vectors and
labels (`write_dekm_study_bundle`). Report seed `s` of the study clusters seed `42 + s`.

Run with a **GPU** accelerator and **Internet on**. If the session would exceed the
12-hour limit, set `SEEDS` to `range(42, 47)` and `range(47, 52)` in two runs.
The result is `/kaggle/working/dekm-fpmax-rcv1.zip`; unzip it into the repository
root: bundles land in `output/dekm_bundles/`, run records in `output/dekm_runs/rcv1/`.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
assert gpus, 'Enable a GPU accelerator in the notebook settings.'

In [ ]:
from pathlib import Path
import subprocess
import sys

DATASET = 'RCV1'
PROTOCOL_ARG = 'release-2021'
SEEDS = range(42, 52)
EXPECTED_REVISION = '80c02e78e695be1ac468255e67cf6ad7a20de62d'

REPO = Path('/kaggle/working/mvdec_fpmax')
subprocess.run(['rm', '-rf', str(REPO)], check=True)
subprocess.run([
    'git', 'clone', '--branch', 'dekm-backbone', '--single-branch',
    'https://github.com/vdat-18/mvdec_fpmax.git', str(REPO),
], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'gdown'], check=True)
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual == EXPECTED_REVISION, actual
print('Repository revision:', actual)

DATASET_ROOT = REPO / 'external_repos/DEKM/datasets'
if DATASET == '20NEWS':
    from sklearn.datasets import fetch_20newsgroups
    fetch_20newsgroups(subset='all', download_if_missing=True)
else:
    subprocess.run([
        sys.executable, str(REPO / 'scripts/fetch_dekm_release.py'), '--datasets', DATASET,
    ], cwd=REPO, check=True)
if DATASET == 'RCV1':
    from sklearn.datasets import fetch_rcv1
    fetch_rcv1(download_if_missing=True)
print(DATASET, 'source is ready.')

In [ ]:
import json
import traceback

from pipeline.study_config import dekm_study_name, load_study_config
from pipeline.study_data import write_dekm_study_bundle

STUDY = load_study_config(REPO / 'configs/fpmax_study.json')
NAME = dekm_study_name(DATASET)
OUTPUT = Path('/kaggle/working/dekm-fpmax-rcv1')
RECORD = OUTPUT / 'output' / 'dekm_runs' / 'rcv1'
RUN_ROOT = RECORD / 'runs'
BUNDLES = OUTPUT / 'output' / 'dekm_bundles'
OUTPUT.mkdir(parents=True, exist_ok=True)
failures = []


def manifest_for(seed):
    found = sorted(RUN_ROOT.glob(f'{DATASET.lower()}/*/*/seed_{seed}/manifest.json'))
    assert len(found) == 1, (seed, found)
    return found[0]


for seed in SEEDS:
    try:
        subprocess.run([
            sys.executable, '-m', 'representation_learning.DEKM_dense', DATASET,
            '--protocol', PROTOCOL_ARG, '--seed', str(seed), '--runs', '1',
            '--require-gpu', '--dataset-root', str(DATASET_ROOT),
            '--output-dir', str(RUN_ROOT),
        ], cwd=REPO, check=True)
        manifest_path = manifest_for(seed)
        bundle = BUNDLES / STUDY.dekm.bundle_name(NAME, seed - STUDY.dekm.embedding_seed_offset)
        write_dekm_study_bundle(
            manifest_path.parent / 'artifact.pkl', DATASET, DATASET_ROOT, bundle,
            n_rows=STUDY.dekm.n_rows,
        )
        print('seed', seed, 'bundle', bundle.name, flush=True)
    except Exception as error:
        failures.append({'seed': seed, 'error_type': type(error).__name__, 'error': str(error)})
        traceback.print_exc()
RECORD.mkdir(parents=True, exist_ok=True)
(RECORD / 'failures.json').write_text(json.dumps(failures, indent=2), encoding='utf-8')
print('Failures:', json.dumps(failures, indent=2))

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from pipeline.study_data import load_dekm_study_bundle

rows = []
for seed in SEEDS:
    if any(f['seed'] == seed for f in failures):
        continue
    manifest = json.loads(manifest_for(seed).read_text(encoding='utf-8'))
    assert manifest['status'] == 'complete' and manifest['method'] == 'DEKM'
    bundle = BUNDLES / STUDY.dekm.bundle_name(NAME, seed - STUDY.dekm.embedding_seed_offset)
    data = load_dekm_study_bundle(bundle, n_rows=STUDY.dekm.n_rows)
    assert int(np.load(bundle)['embedding_seed']) == seed
    rows.append({
        'seed': seed,
        'protocol_id': manifest['protocol_id'],
        'acc': manifest['metrics']['acc'],
        'nmi': manifest['metrics']['nmi'],
        'bundle': bundle.name,
        'bundle_rows': data.n_rows,
        'bundle_sha256': hashlib.sha256(bundle.read_bytes()).hexdigest(),
    })
runs = pd.DataFrame(rows)
runs.to_csv(RECORD / 'dekm_fpmax_runs.csv', index=False)
display(runs)
print('ACC mean', runs['acc'].mean(), 'canonical seeds 42-44: 0.7046')
print('NMI mean', runs['nmi'].mean(), 'canonical seeds 42-44: 0.4594')
assert not failures, failures

In [ ]:
import shutil

for weights in RUN_ROOT.rglob('*.weights.h5'):
    weights.unlink()  # large and not needed by the study; manifests keep their hashes
zip_path = Path(shutil.make_archive(str(OUTPUT), 'zip', str(OUTPUT)))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
zip_path.with_suffix('.zip.sha256').write_text(f'{digest}  {zip_path.name}\n', encoding='ascii')
print('ZIP:', zip_path)
print('SHA-256:', digest)